# Centriole labels colocalisation analysis.

This script will focus on the positional analysis of the centriole markers Centrin2 and CenSpark. The goal is to determine wherether Centrin2 and CenSpark label the centriole correct position or not. If both label are visible at the same location, then we can hypothesis that a centriole has been localized. 

In [1]:
import time
import tifffile
import cv2
import math
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from tqdm import tqdm
from stardist.models import StarDist2D, StarDist3D
from csbdeep.utils import Path, normalize

%matplotlib inline

## Data

In [2]:
IMG_PATH = r"K:\users\voland\Images\Cova_data\ZF_colocalisation\pre_processed\interpolation\interpolated_20260115CFHa_CetnGFP_CS_TL_4-6hpf_e4_t1-7.tif"

In [ ]:
start_time = time.time()
tiff_img = tifffile.imread(IMG_PATH)
print(f"Openning the image took {(time.time() - start_time):.2f} seconds")

Openning the image took 10.05 seconds
Image shape: (7, 80, 3, 768, 768)


In [ ]:
start_time = time.time()
vol = tifffile.memmap(IMG_PATH)
print(f"Openning the image took {(time.time() - start_time):.2f} seconds")

FileNotFoundError: [Errno 2] No such file or directory: 'K:\\users\\voland\\Images\\Cova_data\\ZF_colocalisation\\pre_processed\\interpolation\\interpolated_20260115CFHa_CetnGFP_CS_TL_4-6hpf_e4_t1-7.tif'

## Image analysis

Before going to deep in the semgentation pipeline, we need to analyse our image and better understand it.
Let's first start with some general analysis.

In [ ]:
print(f"Image shape: {vol.shape}")
print(f"  Min value: {vol.min()}")
print(f"  Max value: {vol.max()}")
print(f"  Mean value: {vol.mean():.1f}")

The shape of the dataset corresponds to (T,Z,C,Y,X). 
With the following channel configuration: 
- C0: CenSpark (cyan)
- C1: Cetn2 (green)
- C2: Normal

As the image is in 3D, we can't analyse the entire dataset, however we could take one image that is quite representative of the type of data in it and analyse it.

In [ ]:
img = vol[3,18]
fig, axes = plt.subplots(1,3)
axes[0,0].imshow(img[0], cmap="cyan")
axes[0,1].imshow(img[1],cmap="green_r")
axes[0,2].imshow(img[2], cmap="grays")

In [ ]:
plt.hist(img.flatten(), bins=100, color='blue', alpha=0.7, edgecolor='black')
plt.set_title('Nuclei Intensity Histogram', fontsize=12, fontweight='bold')
plt.set_xlabel('Intensity')
plt.set_ylabel('Frequency')
plt.set_xlim(0, img.max())
plt.axvline(400, color='red', linestyle='--', label='~400 (histogram drop)')
plt.legend()
plt.grid(alpha=0.3)
print(f'Intensity range: [{img.min()} - {img.max()}]')

## 2D segmentation

Different pipelines exist for 2D segmentation. We plan on testing multiple of them (Cellpose, Stardist) and compare there performance.

### Stardist segmentation

In [ ]:
T, Z, C, Y, X = vol.shape
axis_norm = (0,1,2)
norm_volumes = np.empty(vol.shape, vol.dtype)
for t, z, c in product(range(T), range(Z), range(C)):
            img = vol[t,z,c]
            norm_volumes[t,z,c] = normalize(img, 1,99.8, axis=axis_norm)

#### Model loading
We decided to load this model as it is designed for fluorescent images (one detection channel) as we can read in their [website](https://qupath.readthedocs.io/en/stable/docs/deep/stardist.html)

In [ ]:
model = StarDist3D.from_pretrained("dsb2018_heavy_augment")

#### Model running

In [ ]:
norm_vol = norm_volumes[3]
#plt.imshow(norm_vol[0])

##### Segment CenSpark centrioles

In [ ]:
labels, polys = model.predict_instances(
    norm_vol[:,0],  # The image must be normalized
    axes="YXC",
    prob_thresh=0.5,  # Detection probability threshold
    nms_thresh=0.1,  # Remove detections overlapping by more than this threshold
    scale=1,  # Higher values are suitable for lower resolution data
    return_labels=True,
)

# We also get detection probabilities
probabilities = list(polys["prob"])

n_detections = len(probabilities)

print(f'{n_detections} centrioles detected.')

#### Segment Cetn2 centrioles

In [ ]:
labels, polys = model.predict_instances(
    norm_vol[:,1],  # The image must be normalized
    axes="YXC",
    prob_thresh=0.5,  # Detection probability threshold
    nms_thresh=0.1,  # Remove detections overlapping by more than this threshold
    scale=1,  # Higher values are suitable for lower resolution data
    return_labels=True,
)

# We also get detection probabilities
probabilities = list(polys["prob"])

n_detections = len(probabilities)

print(f'{n_detections} centrioles detected.')

### Distance between points

In [ ]:
dist = math.dist(center1, center2)